# **INSTALL & IMPORTS**

In [ ]:
# ── CELL 0: INSTALL & IMPORTS ────────────────────────────────
!pip install ultralytics supervision --quiet
# flash-attn di-install terpisah nanti sebelum YOLOv12 training

import os, json, random, time, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from PIL import Image
from pathlib import Path
from collections import Counter, defaultdict
import supervision as sv
import torch
from ultralytics import YOLO

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
!ls /kaggle/input/datasets/awsaf49/coco-2017-dataset

In [ ]:
!ls /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017

**PATH CONFIG**

In [ ]:
COCO_ROOT = Path('/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017')

TRAIN_IMG = COCO_ROOT / 'train2017'
VAL_IMG   = COCO_ROOT / 'val2017'
TEST_IMG  = COCO_ROOT / 'test2017'
ANN_TRAIN = COCO_ROOT / 'annotations/instances_train2017.json'
ANN_VAL   = COCO_ROOT / 'annotations/instances_val2017.json'

OUT = Path('/kaggle/working')

print("Path check:")
for p in [TRAIN_IMG, VAL_IMG, TEST_IMG, ANN_TRAIN, ANN_VAL]:
    ok = "✓" if p.exists() else "✗ MISSING"
    print(f"  {ok} {p}")

In [ ]:
# ── CELL 2: LOAD ANNOTATIONS ────────────────────────────────
print("Loading annotations...")
with open(ANN_TRAIN) as f: ann_train = json.load(f)
with open(ANN_VAL)   as f: ann_val   = json.load(f)

TRAIN_IMGS  = ann_train['images']
VAL_IMGS    = ann_val['images']
TRAIN_ANNS  = ann_train['annotations']
VAL_ANNS    = ann_val['annotations']
CATEGORIES  = {c['id']: c['name'] for c in ann_train['categories']}
CAT_NAMES   = [c['name'] for c in sorted(ann_train['categories'], key=lambda x: x['id'])]

print(f"\n{'='*48}")
print(f"  Train images     : {len(TRAIN_IMGS):>8,}")
print(f"  Val   images     : {len(VAL_IMGS):>8,}")
print(f"  Train annotations: {len(TRAIN_ANNS):>8,}")
print(f"  Val   annotations: {len(VAL_ANNS):>8,}")
print(f"  Categories (nc)  : {len(CATEGORIES):>8}")
print(f"{'='*48}")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  SECTION A — EDA  (shared, dijalankan sekali)
# ══════════════════════════════════════════════════════════════

# ── CELL 3A: Category Distribution ──────────────────────────
cat_counts = Counter(a['category_id'] for a in TRAIN_ANNS)
cat_df = pd.DataFrame([
    {'category': CATEGORIES[k], 'count': v, 'cat_id': k}
    for k, v in cat_counts.items()
]).sort_values('count', ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
axes[0].barh(cat_df['category'][:20], cat_df['count'][:20], color='#4B2FA0')
axes[0].set_title('Top 20 Categories — Train Annotations')
axes[0].set_xlabel('Count'); axes[0].invert_yaxis()
for i, v in enumerate(cat_df['count'][:20]):
    axes[0].text(v + 200, i, f'{v:,}', va='center', fontsize=8)

axes[1].barh(cat_df['category'][-20:], cat_df['count'][-20:], color='#E05252')
axes[1].set_title('Bottom 20 Categories — Train Annotations')
axes[1].set_xlabel('Count'); axes[1].invert_yaxis()
plt.tight_layout()
plt.savefig(OUT/'eda_01_category_distribution.png', dpi=120)
plt.show()

print(f"Max: {cat_df.iloc[0]['category']} = {cat_df.iloc[0]['count']:,}")
print(f"Min: {cat_df.iloc[-1]['category']} = {cat_df.iloc[-1]['count']:,}")
print(f"Imbalance ratio: {cat_df.iloc[0]['count']/cat_df.iloc[-1]['count']:.1f}x")

In [ ]:
# ── CELL 3B: BBox Size + Aspect Ratio ───────────────────────
areas   = [a['area'] for a in TRAIN_ANNS]
bboxes  = [a['bbox'] for a in TRAIN_ANNS]
bw      = [b[2] for b in bboxes]
bh      = [b[3] for b in bboxes]
ar_box  = [w/h if h > 0 else 1.0 for w, h in zip(bw, bh)]

S = sum(1 for a in areas if a < 32**2)
M = sum(1 for a in areas if 32**2 <= a < 96**2)
L = sum(1 for a in areas if a >= 96**2)
T = len(areas)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(areas, bins=100, color='#4B2FA0', log=True, alpha=0.85)
axes[0].set_xlabel('BBox Area (px²)'); axes[0].set_ylabel('Count (log)')
axes[0].set_title('BBox Area Distribution')

axes[1].hist(bw, bins=80, color='steelblue', alpha=0.6, label='Width')
axes[1].hist(bh, bins=80, color='coral', alpha=0.6, label='Height')
axes[1].set_xlabel('Pixels'); axes[1].set_title('BBox W vs H')
axes[1].legend()

bars = axes[2].bar(['Small\n(<32²)', 'Medium\n(32-96)', 'Large\n(≥96²)'],
                   [S, M, L], color=['#ff6b6b','#ffa94d','#69db7c'],
                   edgecolor='black', linewidth=0.5)
axes[2].set_title('Object Size Distribution'); axes[2].set_ylabel('Count')
for bar, cnt in zip(bars, [S,M,L]):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+500,
                 f'{cnt:,}\n({cnt/T*100:.1f}%)', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUT/'eda_02_bbox_distribution.png', dpi=120)
plt.show()
print(f"Small {S/T*100:.1f}% | Medium {M/T*100:.1f}% | Large {L/T*100:.1f}%")

In [ ]:
# ── CELL 3C: Annotations per Image ──────────────────────────
ann_per_img = Counter(a['image_id'] for a in TRAIN_ANNS)
counts = list(ann_per_img.values())
no_ann = len(TRAIN_IMGS) - len(ann_per_img)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(counts, bins=60, color='teal', alpha=0.8)
axes[0].axvline(np.mean(counts), color='red', linestyle='--',
                label=f'Mean={np.mean(counts):.1f}')
axes[0].axvline(np.median(counts), color='orange', linestyle='--',
                label=f'Median={np.median(counts):.0f}')
axes[0].set_title('Annotations per Image (Train)')
axes[0].set_xlabel('Annotations'); axes[0].legend()

axes[1].bar(['Has Ann', 'No Ann'], [len(ann_per_img), no_ann],
            color=['steelblue','salmon'])
axes[1].set_title('Annotated vs Unannotated')
for i, v in enumerate([len(ann_per_img), no_ann]):
    axes[1].text(i, v+100, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(OUT/'eda_03_annotations_per_image.png', dpi=120)
plt.show()
print(f"Mean={np.mean(counts):.2f} | Median={np.median(counts):.0f} | "
      f"Max={max(counts)} | No-ann images={no_ann:,}")

In [ ]:
# ── CELL 3D: Image Resolution ────────────────────────────────
sample_meta = random.sample(TRAIN_IMGS, 2000)
img_w = [m['width'] for m in sample_meta]
img_h = [m['height'] for m in sample_meta]
img_ar = [w/h for w, h in zip(img_w, img_h)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(img_w, img_h, alpha=0.2, s=4, color='#4B2FA0')
axes[0].axvline(640, color='red', linestyle='--', linewidth=1, label='640px')
axes[0].axhline(640, color='red', linestyle='--', linewidth=1)
axes[0].set_title('Image Resolutions (2K sample)')
axes[0].set_xlabel('Width'); axes[0].set_ylabel('Height')
axes[0].legend()

axes[1].hist(img_ar, bins=60, color='coral', alpha=0.8)
axes[1].axvline(1.0, color='black', linestyle='--', label='Square')
axes[1].axvline(np.mean(img_ar), color='red', linestyle='--',
                label=f'Mean={np.mean(img_ar):.2f}')
axes[1].set_title('Aspect Ratio Distribution')
axes[1].set_xlabel('W/H'); axes[1].legend()

plt.tight_layout()
plt.savefig(OUT/'eda_04_resolution.png', dpi=120)
plt.show()

In [ ]:
# ── CELL 3E: Visual Samples ──────────────────────────────────
def show_samples(img_dir, anns, imgs_meta, cats, n=6, title=""):
    ids = random.sample([m['id'] for m in imgs_meta], n)
    id2file = {m['id']: m['file_name'] for m in imgs_meta}
    ann_map = defaultdict(list)
    for a in anns: ann_map[a['image_id']].append(a)
    colors = plt.cm.tab20(np.linspace(0, 1, 20))

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    for ax, img_id in zip(axes.flat, ids):
        p = img_dir / id2file[img_id]
        if not p.exists(): ax.axis('off'); continue
        ax.imshow(Image.open(p))
        for a in ann_map[img_id]:
            x,y,w,h = a['bbox']
            ci = list(cats.keys()).index(a['category_id']) % 20
            ax.add_patch(patches.Rectangle(
                (x,y), w, h, lw=2, ec=colors[ci], fc='none'))
            ax.text(x, max(y-3,0), cats[a['category_id']],
                    fontsize=7, color='white',
                    bbox=dict(fc=colors[ci], alpha=0.8, pad=1, lw=0))
        ax.axis('off'); ax.set_title(f"ID {img_id}", fontsize=8)
    plt.tight_layout()
    plt.savefig(OUT/f'eda_samples_{title.lower().replace(" ","_")}.png', dpi=100)
    plt.show()

show_samples(TRAIN_IMG, TRAIN_ANNS, TRAIN_IMGS, CATEGORIES, title="Train Samples")
show_samples(VAL_IMG,   VAL_ANNS,   VAL_IMGS,   CATEGORIES, title="Val Samples")

In [ ]:
# ── CELL 3F: EDA Summary ─────────────────────────────────────
print("\n" + "="*55)
print("  EDA SUMMARY — COCO 2017")
print("="*55)
print(f"  Train images      : {len(TRAIN_IMGS):,}")
print(f"  Val images        : {len(VAL_IMGS):,}")
print(f"  Categories        : {len(CATEGORIES)}")
print(f"  Total train anns  : {len(TRAIN_ANNS):,}")
print(f"  Mean anns/image   : {np.mean(counts):.2f}")
print(f"  Small objects     : {S/T*100:.1f}%  (challenge for YOLO)")
print(f"  Medium objects    : {M/T*100:.1f}%")
print(f"  Large objects     : {L/T*100:.1f}%")
print(f"  Class imbalance   : {cat_df.iloc[0]['count']/cat_df.iloc[-1]['count']:.0f}x")
print(f"  Most freq class   : {cat_df.iloc[0]['category']}")
print(f"  Least freq class  : {cat_df.iloc[-1]['category']}")
print("="*55)

**Setup Traning**


In [ ]:
# ── CELL 3G: CONVERT COCO JSON → YOLO TXT ───────────────────
import json
from collections import defaultdict
from pathlib import Path

def coco2yolo(ann_json, label_dir):
    Path(label_dir).mkdir(parents=True, exist_ok=True)
    with open(ann_json) as f:
        data = json.load(f)
    img_info = {img['id']: img for img in data['images']}
    ann_by_img = defaultdict(list)
    for ann in data['annotations']:
        if ann.get('iscrowd', 0): continue
        ann_by_img[ann['image_id']].append(ann)
    cat_ids = sorted([c['id'] for c in data['categories']])
    cat_map  = {cid: i for i, cid in enumerate(cat_ids)}
    converted = 0
    for img_id, anns in ann_by_img.items():
        img = img_info[img_id]
        W, H = img['width'], img['height']
        fname = Path(img['file_name']).stem
        lines = []
        for ann in anns:
            x, y, w, h = ann['bbox']
            cx = (x + w/2) / W
            cy = (y + h/2) / H
            nw = w / W
            nh = h / H
            cid = cat_map[ann['category_id']]
            lines.append(f"{cid} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        with open(f"{label_dir}/{fname}.txt", 'w') as f:
            f.write('\n'.join(lines))
        converted += 1
    print(f"✓ Converted {converted:,} → {label_dir}")

# Cek apakah sudah dikonversi sebelumnya
train_lbl = Path('/kaggle/working/labels/train2017')
val_lbl   = Path('/kaggle/working/labels/val2017')

if train_lbl.exists() and len(list(train_lbl.iterdir())) > 100000:
    print(f"✓ Train labels sudah ada: {len(list(train_lbl.iterdir())):,}")
else:
    print("Converting train (~15 menit)...")
    coco2yolo(str(ANN_TRAIN), str(train_lbl))

if val_lbl.exists() and len(list(val_lbl.iterdir())) > 4000:
    print(f"✓ Val labels sudah ada: {len(list(val_lbl.iterdir())):,}")
else:
    print("Converting val...")
    coco2yolo(str(ANN_VAL), str(val_lbl))

print("Done ✓")

In [ ]:
# ── CELL PENYELAMAT: SYMLINK PER-FILE (KUOTA OUTPUT TETAP 0 GB) ───────────
import os
import glob
import shutil
from pathlib import Path
from tqdm import tqdm

# 1. STOP & HAPUS COPY-AN YANG MEMAKAN KUOTA OUTPUT TADI!
print("1. Membersihkan data copy yang memakan kuota 12GB...")
shutil.rmtree('/kaggle/working/images', ignore_errors=True)
for c in glob.glob('/kaggle/working/**/*.cache', recursive=True):
    try: os.remove(c)
    except: pass
print("   ✓ Kuota Output kembali aman!\n")

# 2. Setup Folder ASLI (Bukan Symlink Folder)
SRC_TRAIN = Path('/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017')
SRC_VAL   = Path('/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/val2017')

DST_TRAIN = Path('/kaggle/working/images/train2017')
DST_VAL   = Path('/kaggle/working/images/val2017')

# Ini membuat folder nyata di working directory
DST_TRAIN.mkdir(parents=True, exist_ok=True)
DST_VAL.mkdir(parents=True, exist_ok=True)

# 3. PROSES SYMLINK FILE (Sangat Cepat & 0 GB)
print("2. Membuat shortcut untuk 118K file Train (Cuma butuh ~30 detik)...")
train_files = list(SRC_TRAIN.glob('*.jpg'))
for f in tqdm(train_files, desc="Symlink Train", unit="file"):
    dst = DST_TRAIN / f.name
    if not dst.exists():
        os.symlink(f, dst)

print("\n3. Membuat shortcut untuk 5K file Val...")
val_files = list(SRC_VAL.glob('*.jpg'))
for f in tqdm(val_files, desc="Symlink Val", unit="file"):
    dst = DST_VAL / f.name
    if not dst.exists():
        os.symlink(f, dst)

print("\n🎉 SELESAI!")

In [ ]:
# STEP 2 — update YAML + hapus cache (Revisi untuk Symlink DDP)
import glob, yaml as _yaml, os
from pathlib import Path

# Bersihkan cache lama
for c in glob.glob('/kaggle/working/**/*.cache', recursive=True):
    try: os.remove(c)
    except: pass
print("Cache cleared ✓")

YAML_PATH = Path('/kaggle/working/coco_direct.yaml')

# --- PERUBAHAN PENTING ADA DI SINI ---
data = {
    'path' : '/kaggle/working',
    'train': 'images/train2017',  # Gunakan path relatif ke symlink yang baru dibuat
    'val'  : 'images/val2017',    # Gunakan path relatif ke symlink yang baru dibuat
    'nc'   : 80,
    'names': CAT_NAMES,
}
# -------------------------------------

with open(YAML_PATH, 'w') as f:
    _yaml.dump(data, f, default_flow_style=False)
    
print(f"YAML → {YAML_PATH}")

In [ ]:
import os

train_imgs = os.listdir(str(TRAIN_IMG))[:5]
train_lbls = os.listdir('/kaggle/working/labels/train2017')[:5]

print("Image files:")
for f in train_imgs: print(f"  {f}")

print("\nLabel files:")
for f in train_lbls: print(f"  {f}")

In [ ]:
# ── CELL 3H: TRANSLATE DICT ──────────────────────────────────
from collections import Counter

COCO_ID = {
    'person': 'orang', 'bicycle': 'sepeda', 'car': 'mobil',
    'motorcycle': 'sepeda motor', 'airplane': 'pesawat', 'bus': 'bus',
    'train': 'kereta api', 'truck': 'truk', 'boat': 'perahu',
    'traffic light': 'lampu lalu lintas', 'fire hydrant': 'hidran kebakaran',
    'stop sign': 'rambu berhenti', 'parking meter': 'meteran parkir',
    'bench': 'bangku', 'bird': 'burung', 'cat': 'kucing',
    'dog': 'anjing', 'horse': 'kuda', 'sheep': 'domba', 'cow': 'sapi',
    'elephant': 'gajah', 'bear': 'beruang', 'zebra': 'zebra',
    'giraffe': 'jerapah', 'backpack': 'ransel', 'umbrella': 'payung',
    'handbag': 'tas tangan', 'tie': 'dasi', 'suitcase': 'koper',
    'frisbee': 'frisbee', 'skis': 'ski', 'snowboard': 'papan salju',
    'sports ball': 'bola olahraga', 'kite': 'layang-layang',
    'baseball bat': 'tongkat bisbol', 'baseball glove': 'sarung tangan bisbol',
    'skateboard': 'skateboard', 'surfboard': 'papan selancar',
    'tennis racket': 'raket tenis', 'bottle': 'botol',
    'wine glass': 'gelas anggur', 'cup': 'cangkir', 'fork': 'garpu',
    'knife': 'pisau', 'spoon': 'sendok', 'bowl': 'mangkuk',
    'banana': 'pisang', 'apple': 'apel', 'sandwich': 'roti lapis',
    'orange': 'jeruk', 'broccoli': 'brokoli', 'carrot': 'wortel',
    'hot dog': 'sosis', 'pizza': 'pizza', 'donut': 'donat',
    'cake': 'kue', 'chair': 'kursi', 'couch': 'sofa',
    'potted plant': 'tanaman pot', 'bed': 'tempat tidur',
    'dining table': 'meja makan', 'toilet': 'toilet', 'tv': 'televisi',
    'laptop': 'laptop', 'mouse': 'tetikus', 'remote': 'remote',
    'keyboard': 'keyboard', 'cell phone': 'ponsel',
    'microwave': 'microwave', 'oven': 'oven',
    'toaster': 'pemanggang roti', 'sink': 'wastafel',
    'refrigerator': 'kulkas', 'book': 'buku', 'clock': 'jam',
    'vase': 'vas bunga', 'scissors': 'gunting',
    'teddy bear': 'boneka beruang', 'hair drier': 'pengering rambut',
    'toothbrush': 'sikat gigi',
}

def translate_labels(labels_en):
    return [COCO_ID.get(l, l) for l in labels_en]

def build_prompt_id(detections):
    if not detections: return "sebuah gambar"
    counts = Counter(translate_labels(detections))
    parts = [f"{cnt} {obj}" if cnt > 1 else f"1 {obj}" for obj, cnt in counts.items()]
    if len(parts) == 1: desc = parts[0]
    elif len(parts) == 2: desc = f"{parts[0]} dan {parts[1]}"
    else: desc = ", ".join(parts[:-1]) + f", dan {parts[-1]}"
    return f"sebuah foto yang menampilkan {desc}"

missing = [n for n in CAT_NAMES if n not in COCO_ID]
print(f"Coverage : {len(COCO_ID)}/80 class")
print(f"Missing  : {missing if missing else 'None ✓'}")
print(f"Test: {build_prompt_id(['car','person','car','dog'])}")

**Yolov11**

In [ ]:
# Tambah ini sebelum model.train()
torch.backends.cudnn.benchmark = True
print(f"VRAM available: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")
print(f"VRAM free: {torch.cuda.memory_reserved(0)/1e9:.1f}GB")

In [ ]:
import psutil
print(f"RAM free: {psutil.virtual_memory().available/1e9:.1f}GB")

In [ ]:
# ══════════════════════════════════════════════════════════════
# SECTION B — YOLOv11
# ══════════════════════════════════════════════════════════════
V11_MODELS = {
    'yolov11n': 'yolo11n.pt',
}
EPOCHS   = 30
IMG_SIZE = 640
BATCH    = 64
NBS      = 128
WORKERS  = 4
DEVICE   = [0, 1]
v11_train_results = {}
v11_val_metrics   = {}
v11_speed_metrics = {}

import gc, torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(0)
torch.cuda.reset_peak_memory_stats(1)

for i in range(2):
    free = torch.cuda.mem_get_info(i)[0] / 1e9
    print(f"GPU {i} free VRAM: {free:.2f} GB")

In [ ]:
# ── CELL 4A: YOLOv11 Model Info ─────────────────────────────
# Jalankan hanya setelah test 1 epoch BERHASIL (Instances > 0)
print("YOLOv11 Model Info\n" + "="*40)
for name, weights in V11_MODELS.items():
    m = YOLO(weights)
    m.model.to('cuda')
    print(f"\n{name}:")
    m.info(verbose=False)
    dummy = torch.zeros(1,3,640,640).to('cuda')
    _ = m.model(dummy)
    mem = torch.cuda.memory_allocated() / 1e6
    print(f"  GPU memory (forward): {mem:.1f} MB")
    del dummy, m; torch.cuda.empty_cache(); gc.collect()

In [ ]:
import ultralytics.data.dataset as _ds
print("Patch aktif?", _ds.YOLODataset.get_labels.__name__)

In [ ]:
# Jalankan ini SEBELUM training, SETELAH patch get_labels
import ultralytics.data.dataset as _ds
import ultralytics.data.utils as _du
from pathlib import Path

_LABEL_ROOT = Path('/kaggle/working/labels')

def _patched_img2label(img_paths):
    out = []
    for p in img_paths:
        p = Path(p)
        split = 'train2017' if 'train2017' in str(p) else 'val2017'
        out.append(str(_LABEL_ROOT / split / (p.stem + '.txt')))
    return out

# Patch di semua lokasi
_du.img2label_paths = _patched_img2label
_ds.img2label_paths = _patched_img2label

print("✓ img2label_paths patched")
print("Test:", _patched_img2label([
    '/kaggle/input/.../train2017/000000001786.jpg'
]))

In [ ]:
print(DEVICE)

In [ ]:
# ── CELL 4C: YOLOv11 Resume dari best.pt ──────────────────
import shutil

def auto_backup(trainer):
    epoch = trainer.epoch + 1
    if epoch % 5 == 0:
        src = Path(trainer.save_dir) / 'weights'
        for f in ['best.pt', 'last.pt']:
            p = src / f
            if p.exists():
                shutil.copy2(p, OUT / f'SAVE_{f}')
                print(f"\n✓ Auto-backup epoch {epoch}: SAVE_{f}")
        csv = Path(trainer.save_dir) / 'results.csv'
        if csv.exists():
            shutil.copy2(csv, OUT / 'SAVE_results.csv')

model = YOLO('/kaggle/input/datasets/claudiojuniarto/yolov11n-bestpt/best.pt')
model.add_callback('on_train_epoch_end', auto_backup)

for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

t0 = time.time()

results = model.train(
    data         = str(YAML_PATH),
    epochs       = 20,
    imgsz        = IMG_SIZE,
    batch        = BATCH,
    nbs          = NBS,
    workers      = WORKERS,
    device       = DEVICE,
    project      = str(OUT / 'runs'),
    name         = 'yolov11n',
    exist_ok     = True,
    seed         = SEED,
    cache        = False,
    amp          = True,
    cos_lr       = True,
    warmup_epochs= 2,
    close_mosaic = 5,
    patience     = 15,
    save_period  = 5,
    verbose      = True,
)

t1 = time.time()

mem_gpus = [torch.cuda.max_memory_allocated(i) / 1e9 for i in range(len(DEVICE))]
avg_peak_mem = sum(mem_gpus) / len(mem_gpus)

v11_train_results['yolov11n'] = {
    'results'        : results,
    'train_time_hrs' : (t1-t0)/3600,
    'peak_vram_gb'   : mem_gpus,
    'avg_vram_gb'    : avg_peak_mem,
}

print(f"\n✓ Selesai!")
print(f"  ⏱ Durasi: {(t1-t0)/3600:.2f} jam")
print(f"  💾 VRAM: {' / '.join([f'{m:.2f}GB' for m in mem_gpus])}")

del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ── CELL 4C: YOLOv11 Full Training (Optimized for Dual T4) ──────────
for model_name, weights in V11_MODELS.items():
    print(f"\n{'='*55}")
    print(f"  [YOLOv11] Training: {model_name}")
    print(f"{'='*55}")

    model = YOLO(weights)

    for i in range(torch.cuda.device_count()):
        torch.cuda.reset_peak_memory_stats(i)

    t0 = time.time()

    results = model.train(
        data         = str(YAML_PATH),
        epochs       = EPOCHS,
        imgsz        = IMG_SIZE,
        batch        = BATCH,
        nbs          = NBS,
        workers      = WORKERS,
        device       = DEVICE,
        project      = str(OUT / 'runs'),
        name         = model_name,
        exist_ok     = True,
        seed         = SEED,
        cache        = False,
        amp          = True,
        cos_lr       = True,
        warmup_epochs= 5,
        multi_scale  = False,
        plots        = True,
        close_mosaic = 10,
        patience     = 15,
        save_period  = 5,
        verbose      = True,
    )

    t1 = time.time()

    mem_gpus = [torch.cuda.max_memory_allocated(i) / 1e9 for i in range(len(DEVICE))]
    avg_peak_mem = sum(mem_gpus) / len(mem_gpus)

    v11_train_results[model_name] = {
        'results'        : results,
        'train_time_hrs' : (t1-t0)/3600,
        'peak_vram_gb'   : mem_gpus,
        'avg_vram_gb'    : avg_peak_mem,
    }

    print(f"\n✓ {model_name} Selesai!")
    print(f"  ⏱ Durasi: {(t1-t0)/3600:.2f} jam")
    print(f"  💾 VRAM (GPU 0/1): {' / '.join([f'{m:.2f}GB' for m in mem_gpus])}")

    del model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# ── CELL 4D: YOLOv11 — Validation ───────────────────────────
for model_name in V11_MODELS.keys():
    best = OUT / 'runs' / model_name / 'weights' / 'best.pt'
    if not best.exists():
        print(f"  ✗ {model_name}: best.pt missing"); continue

    model = YOLO(str(best))
    print(f"\n[YOLOv11] Validating {model_name} on val set...")

    t0 = time.time()
    metrics = model.val(
        data    = str(YAML_PATH),
        imgsz   = IMG_SIZE,
        batch   = BATCH,
        device  = 0,
        verbose = False,
        split   = 'val',
    )
    t1 = time.time()

    # Per-class mAP
    per_class_ap = metrics.box.ap_class_index  # class indices
    per_class_map50 = metrics.box.ap50          # mAP50 per class

    v11_val_metrics[model_name] = {
        'mAP50'          : float(metrics.box.map50),
        'mAP50_95'       : float(metrics.box.map),
        'precision'      : float(metrics.box.mp),
        'recall'         : float(metrics.box.mr),
        'val_time_s'     : t1 - t0,
        'per_class_ap50' : {CAT_NAMES[int(i)]: float(v)
                            for i, v in zip(per_class_ap, per_class_map50)},
    }

    print(f"  mAP@0.5      : {metrics.box.map50:.4f}")
    print(f"  mAP@0.5:0.95 : {metrics.box.map:.4f}")
    print(f"  Precision    : {metrics.box.mp:.4f}")
    print(f"  Recall       : {metrics.box.mr:.4f}")

    del model; torch.cuda.empty_cache(); gc.collect()

In [ ]:
# ── CELL 4E: YOLOv11 — Test Set Inference ───────────────────
# test2017 tidak ada ground truth labels → hanya inference speed
test_imgs = list(VAL_IMG.glob('*.jpg'))[:300]  # pakai val utk speed

for model_name in V11_MODELS.keys():
    best = OUT / 'runs' / model_name / 'weights' / 'best.pt'
    if not best.exists(): continue

    model = YOLO(str(best))

    # Warmup
    for _ in range(5): _ = model(str(test_imgs[0]), verbose=False)

    torch.cuda.synchronize()
    t0 = time.time()
    results = model(test_imgs[:300], verbose=False, stream=True)
    for _ in results: pass
    torch.cuda.synchronize()
    t1 = time.time()

    avg_ms = (t1 - t0) / len(test_imgs) * 1000
    fps    = 1000 / avg_ms

    v11_speed_metrics[model_name] = {
        'avg_inference_ms': round(avg_ms, 2),
        'fps'             : round(fps, 1),
        'n_images'        : len(test_imgs),
    }
    print(f"{model_name}: {avg_ms:.2f} ms | {fps:.1f} FPS")

    del model; torch.cuda.empty_cache(); gc.collect()

In [ ]:
# ── CELL 4F: YOLOv11 — Per-Class mAP Plot ───────────────────
for model_name, metrics in v11_val_metrics.items():
    pc = pd.DataFrame(metrics['per_class_ap50'].items(),
                      columns=['class','ap50']).sort_values('ap50')

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle(f'{model_name} — Per-Class mAP@0.5', fontweight='bold')

    axes[0].barh(pc['class'][-20:], pc['ap50'][-20:], color='#4B2FA0')
    axes[0].set_title('Top 20 Classes'); axes[0].set_xlabel('mAP@0.5')
    axes[0].set_xlim(0, 1)

    axes[1].barh(pc['class'][:20], pc['ap50'][:20], color='#E05252')
    axes[1].set_title('Bottom 20 Classes'); axes[1].set_xlabel('mAP@0.5')
    axes[1].set_xlim(0, 1)

    plt.tight_layout()
    plt.savefig(OUT/f'v11_{model_name}_per_class_map.png', dpi=120)
    plt.show()

In [ ]:
# ── CELL 4G: YOLOv11 — Visual Predictions ───────────────────
def show_predictions(model_path, img_dir, n=6, title=""):
    model = YOLO(str(model_path))
    imgs  = random.sample(list(img_dir.glob('*.jpg')), n)
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle(title, fontsize=12, fontweight='bold')

    for ax, p in zip(axes.flat, imgs):
        res = model(str(p), verbose=False)[0]
        ax.imshow(res.plot(labels=True, conf=True)[:,:,::-1])
        ax.axis('off')
        n_det = len(res.boxes) if res.boxes else 0
        ax.set_title(f"{n_det} detections", fontsize=9)

    plt.tight_layout()
    plt.savefig(OUT/f'{title.replace(" ","_")}_predictions.png', dpi=100)
    plt.show()
    del model; torch.cuda.empty_cache()

for model_name in V11_MODELS.keys():
    best = OUT / 'runs' / model_name / 'weights' / 'best.pt'
    if best.exists():
        show_predictions(best, VAL_IMG, title=f"{model_name} Predictions")

In [ ]:
# ── CELL: Save All YOLOv11 Training Outputs ──────────────────
import shutil, zipfile
from pathlib import Path
from datetime import datetime

OUT = Path('/kaggle/working')

# 1. Copy weights
for model_name in V11_MODELS.keys():
    weights_dir = OUT / 'runs' / model_name / 'weights'
    if not weights_dir.exists():
        print(f"✗ {model_name} weights missing")
        continue
    for w in weights_dir.glob('*.pt'):
        dst = OUT / f'{model_name}_{w.name}'
        shutil.copy2(w, dst)
        print(f"✓ {dst.name}")

# 2. Copy results.csv
for model_name in V11_MODELS.keys():
    csv = OUT / 'runs' / model_name / 'results.csv'
    if csv.exists():
        dst = OUT / f'{model_name}_results.csv'
        shutil.copy2(csv, dst)
        print(f"✓ {dst.name}")

# 3. Copy semua plot dari runs folder
for model_name in V11_MODELS.keys():
    run_dir = OUT / 'runs' / model_name
    for img in run_dir.glob('*.png'):
        dst = OUT / f'{model_name}_{img.name}'
        shutil.copy2(img, dst)
        print(f"✓ {dst.name}")

# 4. Zip semua
timestamp = datetime.now().strftime('%Y%m%d_%H%M')
zip_path  = OUT / f'v11_training_{timestamp}.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT.glob('yolov11*')):
        zf.write(f, f.name)
        print(f"  + {f.name}")
    for f in sorted(OUT.glob('v11_*')):
        zf.write(f, f.name)
        print(f"  + {f.name}")

print(f"\n✓ ZIP: {zip_path.name} ({zip_path.stat().st_size/1e6:.1f} MB)")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  SECTION C — YOLOv12  (Train → Val → Test → Eval)
# ══════════════════════════════════════════════════════════════

# FlashAttention check — T4 = Turing = supported
print("FlashAttention requirement check:")
print(f"  GPU: {torch.cuda.get_device_name(0)}")
cap = torch.cuda.get_device_capability()
print(f"  Compute capability: {cap[0]}.{cap[1]}")
print(f"  FlashAttention supported: {'YES ✓' if cap[0] >= 7 else 'NO — YOLOv12 akan fallback'}")

V12_MODELS = {
    'yolov12n': 'yolo12n.pt',
}

v12_train_results = {}
v12_val_metrics   = {}
v12_speed_metrics = {}

In [ ]:
# ── CELL 5A: YOLOv12 — Model Info ───────────────────────────
print("YOLOv12 Model Info\n" + "="*40)
for name, weights in V12_MODELS.items():
    try:
        m = YOLO(weights)
        print(f"\n{name}:")
        m.info(verbose=False)
        dummy = torch.zeros(1,3,640,640).to('cuda')
        _ = m.model(dummy)
        mem = torch.cuda.memory_allocated() / 1e6
        print(f"  GPU memory (forward): {mem:.1f} MB")
        del dummy, m; torch.cuda.empty_cache(); gc.collect()
    except Exception as e:
        print(f"  ✗ {name}: {e}")

In [ ]:
# ── CELL 5B: YOLOv12 — Training Loop ────────────────────────
for model_name, weights in V12_MODELS.items():
    print(f"\n{'='*55}")
    print(f"  [YOLOv12] Training: {model_name}")
    print(f"{'='*55}")

    model = YOLO(weights)

    for i in range(torch.cuda.device_count()):
        torch.cuda.reset_peak_memory_stats(i)

    t0 = time.time()

    results = model.train(
        data         = str(YAML_PATH),
        epochs       = EPOCHS,
        imgsz        = IMG_SIZE,
        batch        = BATCH,        # 64
        nbs          = NBS,          # 128 ← fix
        workers      = WORKERS,      # 4
        device       = DEVICE,
        project      = str(OUT / 'runs'),
        name         = model_name,
        exist_ok     = True,
        seed         = SEED,
        cache        = False,        # ← fix
        amp          = True,
        cos_lr       = True,         # ← fix
        warmup_epochs= 5,            # ← fix
        multi_scale  = False,
        plots        = True,
        close_mosaic = 10,
        patience     = 15,           # ← fix
        save_period  = 5,            # ← fix
        verbose      = True,
    )

    t1 = time.time()

    mem_gpus = [torch.cuda.max_memory_allocated(i) / 1e9 for i in range(len(DEVICE))]
    avg_peak_mem = sum(mem_gpus) / len(mem_gpus)

    v12_train_results[model_name] = {
        'results'        : results,
        'train_time_hrs' : (t1 - t0) / 3600,
        'peak_vram_gb'   : mem_gpus,
        'avg_vram_gb'    : avg_peak_mem,
    }

    print(f"\n✓ {model_name} Selesai!")
    print(f"  ⏱ Durasi: {(t1-t0)/3600:.2f} jam")
    print(f"  💾 VRAM (GPU 0/1): {' / '.join([f'{m:.2f}GB' for m in mem_gpus])}")

    del model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# ── CELL 5C: YOLOv12 — Training Curves ──────────────────────
# (sama persis dengan 4C, tinggal ganti model_name loop)
for model_name in V12_MODELS.keys():
    results_csv = OUT / 'runs' / model_name / 'results.csv'
    if not results_csv.exists():
        print(f"  ✗ {model_name}: results.csv not found"); continue

    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 4, figsize=(20, 9))
    fig.suptitle(f'{model_name} — Training Curves', fontsize=13, fontweight='bold')

    pairs = [
        ('train/box_loss','val/box_loss','Box Loss'),
        ('train/cls_loss','val/cls_loss','Cls Loss'),
        ('train/dfl_loss','val/dfl_loss','DFL Loss'),
        ('metrics/mAP50(B)','metrics/mAP50-95(B)','mAP'),
        ('metrics/precision(B)', None, 'Precision'),
        ('metrics/recall(B)',    None, 'Recall'),
        ('lr/pg0', None, 'LR pg0'),
        ('lr/pg2', None, 'LR pg2'),
    ]
    for ax, (ct, cv, label) in zip(axes.flat, pairs):
        if ct in df.columns: ax.plot(df['epoch'], df[ct], label='train', color='#2FA07B')
        if cv and cv in df.columns: ax.plot(df['epoch'], df[cv], label='val',
                                            color='coral', linestyle='--')
        ax.set_title(label); ax.set_xlabel('Epoch')
        if cv: ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUT/f'v12_{model_name}_curves.png', dpi=120)
    plt.show()

In [ ]:
# ── CELL 5D: YOLOv12 — Validation ───────────────────────────
for model_name in V12_MODELS.keys():
    best = OUT / 'runs' / model_name / 'weights' / 'best.pt'
    if not best.exists(): print(f"  ✗ {model_name}: best.pt missing"); continue

    model = YOLO(str(best))
    print(f"\n[YOLOv12] Validating {model_name}...")

    t0 = time.time()
    metrics = model.val(
        data=str(YAML_PATH), 
        imgsz=IMG_SIZE,
        batch=BATCH, 
        device=0, 
        verbose=False, 
        split='val',
    )
    t1 = time.time()

    v12_val_metrics[model_name] = {
        'mAP50'          : float(metrics.box.map50),
        'mAP50_95'       : float(metrics.box.map),
        'precision'      : float(metrics.box.mp),
        'recall'         : float(metrics.box.mr),
        'val_time_s'     : t1 - t0,
        'per_class_ap50' : {CAT_NAMES[int(i)]: float(v)
                            for i, v in zip(metrics.box.ap_class_index,
                                            metrics.box.ap50)},
    }
    print(f"  mAP@0.5      : {metrics.box.map50:.4f}")
    print(f"  mAP@0.5:0.95 : {metrics.box.map:.4f}")
    print(f"  Precision    : {metrics.box.mp:.4f}")
    print(f"  Recall       : {metrics.box.mr:.4f}")
    del model; torch.cuda.empty_cache(); gc.collect()

In [ ]:
# ── CELL 5E: YOLOv12 — Speed Benchmark ──────────────────────
for model_name in V12_MODELS.keys():
    best = OUT / 'runs' / model_name / 'weights' / 'best.pt'
    if not best.exists(): continue
    model = YOLO(str(best))

    for _ in range(5): _ = model(str(test_imgs[0]), verbose=False)

    torch.cuda.synchronize()
    t0 = time.time()
    results = model([str(p) for p in test_imgs], verbose=False, stream=True)
    for _ in results: pass
    torch.cuda.synchronize()
    t1 = time.time()

    avg_ms = (t1-t0)/len(test_imgs)*1000
    v12_speed_metrics[model_name] = {
        'avg_inference_ms': round(avg_ms, 2),
        'fps'             : round(1000/avg_ms, 1),
        'n_images'        : len(test_imgs),
    }
    print(f"{model_name}: {avg_ms:.2f} ms | {1000/avg_ms:.1f} FPS")
    del model; torch.cuda.empty_cache(); gc.collect()

In [ ]:
# ── CELL 5F: YOLOv12 — Per-Class + Predictions ──────────────
# Per-class plot
for model_name, metrics in v12_val_metrics.items():
    pc = pd.DataFrame(metrics['per_class_ap50'].items(),
                      columns=['class','ap50']).sort_values('ap50')
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle(f'{model_name} — Per-Class mAP@0.5', fontweight='bold')
    axes[0].barh(pc['class'][-20:], pc['ap50'][-20:], color='#2FA07B')
    axes[0].set_title('Top 20'); axes[0].set_xlabel('mAP@0.5'); axes[0].set_xlim(0,1)
    axes[1].barh(pc['class'][:20], pc['ap50'][:20], color='#E05252')
    axes[1].set_title('Bottom 20'); axes[1].set_xlabel('mAP@0.5'); axes[1].set_xlim(0,1)
    plt.tight_layout()
    plt.savefig(OUT/f'v12_{model_name}_per_class_map.png', dpi=120)
    plt.show()

# Visual predictions
for model_name in V12_MODELS.keys():
    best = OUT/'runs'/model_name/'weights'/'best.pt'
    if best.exists():
        show_predictions(best, VAL_IMG, title=f"{model_name} Predictions")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  SECTION D — FINAL COMPARISON v11 vs v12
# ══════════════════════════════════════════════════════════════

ALL_MODELS = V11_MODELS
ALL_VAL    = v11_val_metrics
ALL_SPEED  = v11_speed_metrics
ALL_TRAIN  = v11_train_results

# ── CELL 6A: Results Table ───────────────────────────────────
rows = []
for m in ALL_MODELS:
    row = {'Model': m, 'Family': 'YOLOv11' if 'v11' in m else 'YOLOv12'}
    if m in ALL_VAL:
        row['mAP@0.5']     = round(ALL_VAL[m]['mAP50'], 4)
        row['mAP@0.5:0.95']= round(ALL_VAL[m]['mAP50_95'], 4)
        row['Precision']   = round(ALL_VAL[m]['precision'], 4)
        row['Recall']      = round(ALL_VAL[m]['recall'], 4)
    if m in ALL_SPEED:
        row['ms/img']      = ALL_SPEED[m]['avg_inference_ms']
        row['FPS']         = ALL_SPEED[m]['fps']
    if m in ALL_TRAIN:
        row['Train(hr)']   = round(ALL_TRAIN[m]['train_time_hrs'], 2)
        vram = ALL_TRAIN[m]['peak_vram_gb']
        row['Peak VRAM(GB)'] = round(vram if isinstance(vram, float) else max(vram), 2)
    rows.append(row)

df_final = pd.DataFrame(rows)
print("\n" + "="*80)
print("FINAL BENCHMARK — YOLOv11 vs YOLOv12 | COCO 2017 | 50 Epochs")
print("="*80)
print(df_final.to_string(index=False))
df_final.to_csv(OUT/'final_benchmark.csv', index=False)
print(f"\nSaved → {OUT/'final_benchmark.csv'}")

In [ ]:
# ── CELL 6B: Bar Chart All Metrics ──────────────────────────
models_list = list(ALL_VAL.keys())
colors_v11  = ['#4B2FA0', '#7B5FD0']
colors_v12  = ['#2FA07B', '#5FD0A0']
bar_colors  = colors_v11 + colors_v12

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('YOLOv11 vs YOLOv12 — COCO 2017 Full Comparison', fontsize=14, fontweight='bold')

metric_specs = [
    ('mAP50',    'mAP@0.5',       ALL_VAL),
    ('mAP50_95', 'mAP@0.5:0.95',  ALL_VAL),
    ('precision','Precision',      ALL_VAL),
    ('recall',   'Recall',         ALL_VAL),
    ('avg_inference_ms', 'Inference (ms) ↓', ALL_SPEED),
    ('fps',      'FPS ↑',          ALL_SPEED),
]

for ax, (key, label, src) in zip(axes.flat, metric_specs):
    vals = [src[m][key] for m in models_list if m in src]
    mods = [m for m in models_list if m in src]
    bars = ax.bar(mods, vals, color=bar_colors[:len(mods)],
                  edgecolor='black', linewidth=0.4)
    ax.set_title(label, fontweight='bold')
    ax.set_xticklabels(mods, rotation=20, ha='right', fontsize=9)
    ax.grid(True, alpha=0.2, axis='y')
    ypad = max(vals) * 0.04
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+ypad,
                f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(OUT/'comparison_all_metrics.png', dpi=150)
plt.show()

In [ ]:
# ── CELL 6C: Speed vs Accuracy Scatter ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Speed vs Accuracy Trade-off', fontsize=13, fontweight='bold')

for ax, (y_key, y_label) in zip(axes, [
    ('mAP50', 'mAP@0.5'),
    ('mAP50_95', 'mAP@0.5:0.95')
]):
    for m, color in zip(models_list, bar_colors):
        if m in ALL_VAL and m in ALL_SPEED:
            x = ALL_SPEED[m]['avg_inference_ms']
            y = ALL_VAL[m][y_key]
            ax.scatter(x, y, s=250, color=color, zorder=5, label=m, edgecolors='black')
            ax.annotate(m, (x, y), xytext=(5, 5),
                        textcoords='offset points', fontsize=9)
    ax.set_xlabel('Inference (ms) — lower better →')
    ax.set_ylabel(f'{y_label} — higher better ↑')
    ax.set_title(y_label); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUT/'speed_accuracy_scatter.png', dpi=150)
plt.show()

In [ ]:
# ── CELL 6D: Training Curves Overlay (v11s vs v12s) ─────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training Curves Overlay — v11n vs v12n (nano)', fontweight='bold')

models_to_compare = ['yolov11n', 'yolov12n']

metrics_overlay = [
    ('metrics/mAP50(B)', 'mAP@0.5'),
    ('train/box_loss',   'Box Loss'),
    ('train/cls_loss',   'Cls Loss'),
]

color_map = {
    'yolov11n': '#4B2FA0',
    'yolov12n': '#2FA07B',   # fix
}

for ax, (col, label) in zip(axes, metrics_overlay):
    for m in models_to_compare:
        csv = OUT / 'runs' / m / 'results.csv'
        if not csv.exists():
            continue
        df = pd.read_csv(csv)
        df.columns = df.columns.str.strip()
        if col in df.columns:
            ax.plot(df['epoch'], df[col], label=m, color=color_map[m], linewidth=2)
    ax.set_title(label)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUT / 'overlay_curves_nano.png', dpi=120)
plt.show()

In [ ]:
# ── CELL 6E: Winner + Recommendation ────────────────────────
print("\n" + "="*60)
print("  WINNER ANALYSIS — Deployment Recommendation")
print("="*60)

# Composite score: normalize 4 metrics, equal weight
metrics_for_score = ['mAP50', 'mAP50_95', 'precision', 'recall']
scores = {}

for m in models_list:
    if m not in ALL_VAL or m not in ALL_SPEED: continue
    norm_scores = []
    for k in metrics_for_score:
        vals_all = [ALL_VAL[x][k] for x in models_list if x in ALL_VAL]
        norm_scores.append(ALL_VAL[m][k] / max(vals_all))

    # FPS (normalized, higher better)
    fps_all = [ALL_SPEED[x]['fps'] for x in models_list if x in ALL_SPEED]
    norm_scores.append(ALL_SPEED[m]['fps'] / max(fps_all))

    scores[m] = np.mean(norm_scores)

winner = max(scores, key=scores.get)

print(f"\n  Composite Score (mAP50 + mAP50-95 + Prec + Recall + FPS):")
for m, s in sorted(scores.items(), key=lambda x: -x[1]):
    tag = "  ← WINNER ✓" if m == winner else ""
    print(f"    {m:<15}: {s:.4f}{tag}")

print(f"\n  → RECOMMENDED MODEL: {winner}")

if winner:
    v = ALL_VAL[winner]; sp = ALL_SPEED[winner]
    print(f"     mAP@0.5      : {v['mAP50']:.4f}")
    print(f"     mAP@0.5:0.95 : {v['mAP50_95']:.4f}")
    print(f"     Precision    : {v['precision']:.4f}")
    print(f"     Recall       : {v['recall']:.4f}")
    print(f"     Inference    : {sp['avg_inference_ms']:.2f} ms | {sp['fps']:.1f} FPS")

print(f"\n  Output files:")
for f in sorted(OUT.glob('*.png')) : print(f"    {f.name}")
for f in sorted(OUT.glob('*.csv')) : print(f"    {f.name}")
print("="*60)

In [ ]:
# ── CELL 7: Save All Outputs ─────────────────────────────────
import shutil
import zipfile
from datetime import datetime

print("Saving all outputs...")

# 1. Save final benchmark CSV (sudah ada)
print(f"  ✓ final_benchmark.csv")

# 2. Save semua weights
for model_name in {**V11_MODELS, **V12_MODELS}.keys():
    weights_dir = OUT / 'runs' / model_name / 'weights'
    if weights_dir.exists():
        for w in weights_dir.glob('*.pt'):
            dst = OUT / f'{model_name}_{w.name}'
            shutil.copy2(w, dst)
            print(f"  ✓ {dst.name}")

# 3. Save semua results.csv per model
for model_name in {**V11_MODELS, **V12_MODELS}.keys():
    csv = OUT / 'runs' / model_name / 'results.csv'
    if csv.exists():
        dst = OUT / f'{model_name}_results.csv'
        shutil.copy2(csv, dst)
        print(f"  ✓ {dst.name}")

# 4. Zip semua output jadi 1 file
timestamp = datetime.now().strftime('%Y%m%d_%H%M')
zip_path  = OUT / f'yolo_comparison_{timestamp}.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # PNG plots
    for f in sorted(OUT.glob('*.png')):
        zf.write(f, f.name)
        print(f"  + {f.name}")
    # CSV files
    for f in sorted(OUT.glob('*.csv')):
        zf.write(f, f.name)
        print(f"  + {f.name}")
    # Weights
    for f in sorted(OUT.glob('*.pt')):
        zf.write(f, f.name)
        print(f"  + {f.name}")

zip_size = zip_path.stat().st_size / 1e6
print(f"\n✓ ZIP saved: {zip_path.name} ({zip_size:.1f} MB)")
print(f"  Download dari Kaggle Output tab → {zip_path.name}")

# 5. Print semua file output
print(f"\n{'='*50}")
print("  ALL OUTPUT FILES")
print(f"{'='*50}")
for f in sorted(OUT.iterdir()):
    if f.is_file():
        size = f.stat().st_size / 1e6
        print(f"  {f.name:<45} {size:.1f} MB")